# 04_step2_F1_applied_all_GCM - Apply F1 to all GCM runs (global, 1 deg)   [PLAN A]

The **same** F1 function (parameters from step1) is applied to:
EC-Earth3 historical, GCM 2025 / 2050, hosing / no-hosing, all already regridded to 1 deg (notebook 02).

The "ERA5-like 1 deg GCM" of the **2025 no-hosing** run is the starting point of the downscaling (step4).

One file per run: `ba_1deg/<run>_F1.nc` (variables `pr`, `tasmax`, `tasmin`).

In [ ]:
import os
os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import sys, warnings
from pathlib import Path
for _p in (Path.cwd(), Path.cwd().parent):   # ba_utils.py: same folder as the notebook, or the parent folder
    sys.path.insert(0, str(_p))
import numpy as np, xarray as xr, pandas as pd
import matplotlib.pyplot as plt
import ba_utils as bu
warnings.filterwarnings("ignore")

CONFIG_PATH = None   # None = config_<profile>.json found automatically (written by notebook 00)
USE_DASK_CLUSTER = False   # True = use several CPU cores (recommended on a JEO-lab / JEO-desk node)
cfg = bu.load_config(CONFIG_PATH)
CLIENT = bu.start_dask() if USE_DASK_CLUSTER else None
VARS = cfg["variables"]                      # ['pr', 'tasmax', 'tasmin']
DOMAIN = bu.rcm_domain(cfg)   # RCM domain for regional diagnostics (None = global until the RCM path is filled in)

APPLY_TO = ["gcm_historical", "gcm_2025_nohosing", "gcm_2025_hosing", "gcm_2050_nohosing", "gcm_2050_hosing"]
CHUNKS_SPACE = {"lat": 30, "lon": 60}
OVERWRITE = False
REGRID_DIR = bu.out_dir(cfg, "plan_A", "regrid_1deg")
F1_DIR = bu.out_dir(cfg, "plan_A", "F1")
BA1_DIR = bu.out_dir(cfg, "plan_A", "ba_1deg")
assert (F1_DIR / "qdm_pr.nc").exists(), "Run notebook step1 first (F1 not calibrated)."

## 1. Application

In [ ]:
for name in APPLY_TO:
    out_path = BA1_DIR / f"{name}_F1.nc"
    if out_path.exists() and not OVERWRITE:
        print(f"[already done] {name}"); continue
    missing = [v for v in VARS if not (REGRID_DIR / f"{name}_{v}_1deg.nc").exists()]
    if missing:
        print(f"[missing] {name}: {missing} (notebook 02)"); continue
    print(f"-> {name}")
    sim = {v: bu.read_processed(REGRID_DIR / f"{name}_{v}_1deg.nc", v) for v in VARS}
    adj = bu.apply_F(sim, F1_DIR, chunks_space=CHUNKS_SPACE)
    bu.save_ds(xr.Dataset(adj), out_path)

## 2. Quick checks

- no negative precipitation, `tasmin < tasmax`, no new NaN;
- is the GCM **signal** preserved? (change 2025 no-hosing - historical, and hosing - no-hosing, before / after F1,
  monthly means over the RCM domain, global until the RCM path is filled in). QDM is designed to preserve the change quantile by quantile;
  the "raw" and "adjusted" curves should have the same shape (not necessarily the same level for `pr`, which is relative).

In [ ]:
rows = []
for name in APPLY_TO:
    f = BA1_DIR / f"{name}_F1.nc"
    if not f.exists():
        continue
    ds = xr.open_dataset(f, chunks={"time": 365})
    s = ds.isel(time=slice(0, 365 * 3))
    rows.append({"run": name,
                 "pr min": float(s["pr"].min()), "pr max": float(s["pr"].max()),
                 "frac tasmin>=tasmax (should be 0)": float(((s["tasmin"] >= s["tasmax"]) & s["tasmax"].notnull()).mean()),
                 "frac NaN (pr)": float(s["pr"].isnull().mean())})
pd.DataFrame(rows)

In [ ]:
def monthly_domain(da):
    return bu.area_mean(da, DOMAIN).groupby("time.month").mean().compute()

def load_pair(name, var):
    f_raw = REGRID_DIR / f"{name}_{var}_1deg.nc"; f_adj = BA1_DIR / f"{name}_F1.nc"
    if not (f_raw.exists() and f_adj.exists()):
        return None, None
    return bu.read_processed(f_raw, var), bu.read_processed(f_adj, var)

comparisons = [("gcm_2025_nohosing", "gcm_historical", "2025 no-hosing - historical"),
               ("gcm_2025_hosing", "gcm_2025_nohosing", "2025 hosing - no-hosing")]
fig, axes = plt.subplots(len(comparisons), 3, figsize=(16, 7))
for r, (a, b, lab) in enumerate(comparisons):
    for c, var in enumerate(VARS):
        ra, aa = load_pair(a, var); rb, ab = load_pair(b, var)
        ax = axes[r, c]
        if ra is None or rb is None:
            ax.set_title(f"{lab} / {var}: missing"); continue
        raw = monthly_domain(ra) - monthly_domain(rb)
        adj = monthly_domain(aa) - monthly_domain(ab)
        ax.plot(raw.month, raw, "o-", label="raw"); ax.plot(adj.month, adj, "s--", label="after F1")
        ax.axhline(0, color="grey", lw=0.5); ax.set_title(f"{lab} - {var}"); ax.set_xlabel("month")
axes[0, 0].legend()
plt.tight_layout(); plt.savefig(bu.out_dir(cfg, "plan_A", "F1") / "signal_before_after.png", dpi=110); plt.show()